# 💵 Predictor de flujo de efectivo con modelos de Hugging Face
### Chronos-2 · Chronos-Bolt Small · ETS / SARIMAX

**Objetivo.** Pronosticar el flujo neto diario de efectivo a partir de los movimientos bancarios (export FEBAN de depósitos cargados) y comparar dos modelos fundacionales de series de tiempo publicados en Hugging Face contra dos modelos estadísticos clásicos, midiendo exactitud, capacidad de alerta y latencia.

| Modelo | Familia | Fuente | Cómo aprende |
|---|---|---|---|
| **Chronos-2** | Fundacional (multiserie, admite covariables) | `amazon/chronos-2` | *Zero-shot*: no se reentrena; usa la historia como contexto |
| **Chronos-2 + temporada** | Chronos-2 con la covariable `temporada` conocida a futuro (`future_df`) | `amazon/chronos-2` | *Zero-shot*; aprende en contexto la relación con la covariable |
| **Chronos-2 (cross-learning)** | La misma red pronosticando todas las series en conjunto | `amazon/chronos-2` | Comparte información entre las series del lote |
| **Chronos-Bolt Small** | Fundacional (T5 encoder-decoder, univariado) | `amazon/chronos-bolt-small` | *Zero-shot* |
| **ETS** | Suavizamiento exponencial con error aditivo | `statsmodels` | Máxima verosimilitud, por serie |
| **SARIMAX** | ARIMA estacional con regresores exógenos (calendario) | `statsmodels` | Máxima verosimilitud, por serie |
| **SARIMAX + temporada** | SARIMAX con la covariable `temporada` como `exog` | `statsmodels` | Máxima verosimilitud, por serie |
| **Naive** | Referencia: mañana = hoy | — | — |

La covariable **`temporada`** son dos indicadores conocidos de antemano: ingreso alto (enero a marzo) y gasto alto (septiembre a diciembre). Comparar cada modelo con y sin ella muestra si anticipa el cambio de temporada o lo detecta tarde.

**Cómo usarlo en Colab**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (también corre en CPU).
2. Opcional: guarda tu token de Hugging Face en **Secrets (🔑)** con el nombre `HF_TOKEN`.
3. *Entorno de ejecución → Ejecutar todas*. El cuaderno busca `DEPOSITOS_CARGADOS_FEBAN_sep2025_sep2026_anon.xlsx` junto a él, en `data/` y en `../data/` (estructura del repositorio). Si no lo encuentra, pide subirlo; sirve cualquier export FEBAN con las mismas columnas.
4. Horizonte, orígenes del backtest, cuantiles, temporadas y reglas de tesorería se ajustan en la celda **4.1**.

Con un año de historia, horizonte de 5 días y todos los orígenes, la corrida completa tarda del orden de 15 a 30 minutos en CPU. Casi todo ese tiempo es de SARIMAX en CPU; Chronos en GPU tarda pocos minutos.

**Contenido**
1. Configuración de GPU
2. Instalación de dependencias
3. Autenticación en Hugging Face Hub
4. Carga de modelos y datasets
5. Pipeline de inferencia
6. Cálculo de métricas (accuracy, precision, recall, F1-score)
7. Medición de latencia
8. Tablas y resúmenes
9. Conclusiones parciales

## 1. Configuración de GPU
Detecta la GPU, elige el tipo numérico de inferencia y fija semillas. En una **T4** (compute capability 7.5) se usa `float32`, porque no tiene soporte nativo de `bfloat16`; en **L4/A100** se usa `bfloat16`. La función `sincronizar()` hace que las mediciones de latencia esperen a que la GPU termine de verdad.

In [ ]:
import os, time, random, platform, subprocess, warnings
import numpy as np
import torch


def _nvidia_smi():
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20,
        )
        return r.stdout.strip() or r.stderr.strip()
    except FileNotFoundError:
        return "no disponible (sin GPU NVIDIA en este entorno)"


print("Python     :", platform.python_version())
print("PyTorch    :", torch.__version__, "| CUDA:", torch.version.cuda)
print("nvidia-smi :", _nvidia_smi())

if torch.cuda.is_available():
    DEVICE = "cuda"
    capacidad = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    TORCH_DTYPE = torch.bfloat16 if capacidad[0] >= 8 else torch.float32
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU        : {props.name} | VRAM {props.total_memory / 1024**3:.1f} GB "
          f"| compute capability {capacidad[0]}.{capacidad[1]}")
else:
    DEVICE = "cpu"
    TORCH_DTYPE = torch.float32
    print("⚠️  Sin GPU: el cuaderno funciona en CPU (los modelos son pequeños), pero la latencia no será representativa.\n"
          "    En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4.")

print("Dispositivo:", DEVICE, "| dtype de inferencia:", TORCH_DTYPE)

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEMILLA)


def sincronizar():
    """Espera a que terminen los kernels de GPU para que el cronómetro mida tiempo real."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 2. Instalación de dependencias
Colab ya trae `torch`, `pandas`, `statsmodels`, `scikit-learn` y `matplotlib`. Se instala `chronos-forecasting` (la librería oficial de Amazon que contiene Chronos-2 y Chronos-Bolt) y `openpyxl` para leer el Excel.

In [ ]:
%pip install -q "chronos-forecasting>=2.3,<3" "statsmodels>=0.14" "scikit-learn>=1.3" openpyxl

In [ ]:
import importlib.metadata as metadata

for paquete in ["chronos-forecasting", "transformers", "accelerate", "torch", "huggingface_hub",
                "pandas", "numpy", "statsmodels", "scikit-learn", "openpyxl"]:
    try:
        print(f"{paquete:22s} {metadata.version(paquete)}")
    except metadata.PackageNotFoundError:
        print(f"{paquete:22s} NO INSTALADO")

## 3. Autenticación en Hugging Face Hub
Los dos modelos son públicos, así que el token es opcional: autenticarse evita los límites de descarga anónima y deja la sesión lista si después publicas un modelo ajustado. Crea un token de **lectura** en *huggingface.co → Settings → Access Tokens* y guárdalo en los **Secrets** de Colab como `HF_TOKEN`, con acceso activado para este cuaderno.

In [ ]:
from huggingface_hub import login, whoami, model_info

REPOS_HF = {
    "Chronos-2": "amazon/chronos-2",
    "Chronos-Bolt Small": "amazon/chronos-bolt-small",
}

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"No se encontró HF_TOKEN en los Secrets de Colab ({type(e).__name__}).")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Sesión iniciada en Hugging Face Hub como:", whoami(token=HF_TOKEN)["name"])
else:
    HF_TOKEN = None
    print("ℹ️  Sin token: los modelos públicos se descargarán de forma anónima.")

for nombre, repo in REPOS_HF.items():
    try:
        info = model_info(repo, token=HF_TOKEN)
        licencia = getattr(info.card_data, "license", None) if info.card_data else None
        print(f"   {nombre:20s} → {repo:28s} revisión {info.sha[:8]} | licencia: {licencia or 'n/d'}")
    except Exception as e:
        print(f"   ⚠️ No se pudo consultar {repo}: {e}")

## 4. Carga de modelos y datasets
### 4.1 Parámetros del experimento
Todo lo configurable está aquí. Si cambias algo, vuelve a ejecutar desde esta celda.

In [ ]:
# ---------- Archivo de entrada ----------
ARCHIVO_EXCEL = "DEPOSITOS_CARGADOS_FEBAN_sep2025_sep2026_anon.xlsx"
HOJA_EXCEL = 0
MONEDA = "MXN"

# Nombre interno → columna del export FEBAN
COLUMNAS = {
    "fecha": "Value date",
    "monto": "Amount",
    "banco": "House Bk",
    "linea_extracto": "MR no",            # número de partida dentro del extracto
    "concepto": "Payment Notes",
    "tipo_tx": "External transaction",
    "moneda": "AC",
    "clave_extracto": "Shrt key",         # una clave por cada carga del extracto
    "regla_contable": "Post. rule",
}

# ---------- Limpieza ----------
DEDUPLICAR = True                         # protege contra exports con varias cargas del mismo extracto
LLAVE_MOVIMIENTO = ["banco", "linea_extracto", "concepto", "monto", "fecha"]

# ---------- Clasificación: (categoría, regex sobre el concepto, tipos de transacción, ¿es tesorería?) ----------
# La primera regla que coincide gana. Tesorería = movimientos entre cuentas propias (no son flujo del negocio).
# Ajusta los patrones con la lista real de tus cuentas propias.
REGLAS_CATEGORIA = [
    ("Tesorería: barrido (cash pooling)",    r"SWEEP",                                 [],       True),
    ("Tesorería: giro / traspaso de fondos", r"GIROFONDO|SPEI ENVIADO HSB",            [],       True),
    ("Comisiones e IVA bancario",            r"COMISION|PENA|I\.V\.A|TRANSFENVSPEI",   ["NCOM"], False),
    ("Compensación SPEI",                    r"COMPENSACION",                          [],       False),
    ("Cargo por cartera",                    r"CARTERA",                               [],       False),
    ("Depósitos (efectivo / terceros)",      r"DEPOSITO",                              ["NCOL"], False),
    ("Pago cuenta de tercero",               r"PAGO CUENTA DE T",                      [],       False),
]
CATEGORIA_INGRESO_DEFAULT = "Cobranza SPEI / TEF"
CATEGORIA_EGRESO_DEFAULT = "Otros egresos"
SERIES_POR_BANCO = True                   # además del total, una serie operativa por banco

# ---------- Pronóstico y evaluación ----------
FRECUENCIA = "B"                          # días hábiles (lunes a viernes)
HORIZONTE = 5                             # días hábiles por pronóstico en el backtest
HORIZONTE_FINAL = 5                       # días hábiles del pronóstico hacia adelante
MIN_CONTEXTO = 3                          # días de historia mínimos antes del primer origen
MAX_ORIGENES = 280                        # orígenes walk-forward a evaluar (los más recientes; 280 ≈ todo el año)
CUANTILES = [0.1, 0.5, 0.9]               # P10–P90 = intervalo del 80 %
ESTACIONALIDAD = 5                        # ciclo semanal en días hábiles
USAR_EXOGENAS_SARIMAX = True              # calendario como X de SARIMAX (solo con ≥ 30 días)
SARIMAX_REESTIMAR_CADA = 5                # backtest: re-estima SARIMAX cada N orígenes; entre tanto filtra con los
                                          # últimos parámetros (1 = re-estimar en cada origen, más lento)

# ---------- Covariable temporada (conocida a futuro) ----------
MESES_INGRESO_ALTO = (1, 2, 3)            # enero a marzo
MESES_GASTO_ALTO = (9, 10, 11, 12)        # septiembre a diciembre
MIN_DIAS_TEMPORADA = 10                   # días hábiles mínimos de una temporada en la historia para usarla
FECHA_CORTE = "2026-09-01"                # separa las métricas del backtest: antes / desde esta fecha

# ---------- Latencia ----------
N_REPETICIONES_LATENCIA = 20              # se recorta a ≥ 5 si un modelo acumula más de 30 s
TAMANOS_LOTE = [1, 16, 64]                # número de series para la prueba de escalabilidad

### 4.2 Carga del Excel de movimientos bancarios
Busca el archivo junto al cuaderno, en `data/` y en `../data/`. Si no está, se abre el selector de archivos de Colab.

In [ ]:
import re
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 30)

candidatas = [Path(ARCHIVO_EXCEL), Path("data") / ARCHIVO_EXCEL, Path("../data") / ARCHIVO_EXCEL,
              Path("predictor-flujo-efectivo/data") / ARCHIVO_EXCEL]          # local, repositorio o clon en Colab
ruta_excel = next((p for p in candidatas if p.exists()), None)
if ruta_excel is None:
    try:
        from google.colab import files
        print(f"No encontré '{ARCHIVO_EXCEL}'. Selecciona el Excel de movimientos bancarios:")
        subidos = files.upload()
        ruta_excel = Path(next(iter(subidos)))
    except ImportError:
        raise FileNotFoundError(f"Coloca '{ARCHIVO_EXCEL}' junto al cuaderno o ajusta ARCHIVO_EXCEL en 4.1.")

encabezado = pd.read_excel(ruta_excel, sheet_name=HOJA_EXCEL, nrows=0).columns
como_texto = {COLUMNAS[k]: str for k in ("linea_extracto", "clave_extracto") if COLUMNAS[k] in encabezado}
crudo = pd.read_excel(ruta_excel, sheet_name=HOJA_EXCEL, dtype=como_texto)

requeridas = ["fecha", "monto", "banco", "concepto"]
faltan = [COLUMNAS[k] for k in requeridas if COLUMNAS[k] not in crudo.columns]
if faltan:
    raise KeyError(f"Faltan columnas obligatorias {faltan}. Columnas del archivo: {list(crudo.columns)}")

crudo = crudo.rename(columns={v: k for k, v in COLUMNAS.items() if v in crudo.columns})
crudo["fecha"] = pd.to_datetime(crudo["fecha"], dayfirst=True).dt.normalize().astype("datetime64[ns]")
crudo["monto"] = pd.to_numeric(crudo["monto"], errors="coerce")
crudo["banco"] = crudo["banco"].astype(str).str.strip().str.upper()

sin_monto = int(crudo["monto"].isna().sum())
if sin_monto:
    print(f"⚠️ {sin_monto} filas sin monto numérico se descartan.")
    crudo = crudo.dropna(subset=["monto"])
if "moneda" in crudo.columns:
    monedas = crudo["moneda"].value_counts()
    if len(monedas) > 1 or MONEDA not in monedas.index:
        print("⚠️ Monedas encontradas:", monedas.to_dict(), "→ se conserva solo", MONEDA)
    crudo = crudo[crudo["moneda"] == MONEDA].copy()

print(f"Archivo : {ruta_excel.name}")
print(f"Filas   : {len(crudo):,} | columnas: {crudo.shape[1]}")
print(f"Periodo : {crudo['fecha'].min():%d-%m-%Y} → {crudo['fecha'].max():%d-%m-%Y}")
print("Bancos  :", crudo["banco"].value_counts().to_dict())
display(crudo.head(8))

### 4.3 Limpieza: deduplicación del export
Un export FEBAN puede traer el mismo extracto cargado varias veces (una `Shrt key` por carga), y entonces cada partida aparece repetida. Si se suman las filas tal cual, el flujo queda inflado tantas veces como cargas haya. Un movimiento único se identifica por **banco + número de partida (`MR no`) + concepto + monto + fecha valor**. Con un archivo que ya trae una sola carga por extracto, este paso no encuentra duplicados y solo sirve de protección.

In [ ]:
llave = [c for c in LLAVE_MOVIMIENTO if c in crudo.columns]
copias = crudo.groupby(llave, dropna=False).size()

resumen_dedup = crudo.groupby("banco").agg(filas_export=("monto", "size"))
if "clave_extracto" in crudo.columns:
    resumen_dedup["cargas_del_extracto"] = crudo.groupby("banco")["clave_extracto"].nunique()

mov = (crudo.drop_duplicates(subset=llave) if DEDUPLICAR else crudo).copy().reset_index(drop=True)

resumen_dedup["movimientos_unicos"] = mov.groupby("banco").size()
resumen_dedup["neto_export (M MXN)"] = crudo.groupby("banco")["monto"].sum() / 1e6
resumen_dedup["neto_real (M MXN)"] = mov.groupby("banco")["monto"].sum() / 1e6
resumen_dedup.loc["TOTAL"] = resumen_dedup.sum(numeric_only=True)
if "cargas_del_extracto" in resumen_dedup:
    resumen_dedup.loc["TOTAL", "cargas_del_extracto"] = np.nan
resumen_dedup.insert(
    resumen_dedup.columns.get_loc("movimientos_unicos") + 1, "veces_repetido",
    resumen_dedup["filas_export"] / resumen_dedup["movimientos_unicos"],
)

print(f"{len(crudo):,} filas → {len(mov):,} movimientos únicos  (llave: {', '.join(llave)})")
print("Distribución de repeticiones por movimiento:", copias.value_counts().sort_index().to_dict())
if not DEDUPLICAR:
    print("⚠️ DEDUPLICAR = False: el flujo diario quedará inflado.")
display(resumen_dedup.style.format(
    {"filas_export": "{:,.0f}", "cargas_del_extracto": "{:,.0f}", "movimientos_unicos": "{:,.0f}",
     "veces_repetido": "{:,.1f}", "neto_export (M MXN)": "{:,.2f}", "neto_real (M MXN)": "{:,.2f}"},
    na_rep="—"))

### 4.4 Clasificación: flujo operativo vs. tesorería
Los barridos de *cash pooling* y los giros entre cuentas propias no son flujo del negocio: son decisiones de tesorería que mueven efectivo entre cuentas y suelen ser de un orden de magnitud mayor que la cobranza. Se separan para que los modelos aprendan el patrón **operativo** (cobranza − comisiones − cargos). Revisa las reglas de la celda 4.1 con tu lista de cuentas propias.

In [ ]:
def clasificar(concepto, tipo, monto):
    texto, tipo = str(concepto).upper(), str(tipo).upper()
    for categoria, patron, tipos, es_tesoreria in REGLAS_CATEGORIA:
        if re.search(patron, texto) or tipo in tipos:
            return categoria, es_tesoreria
    return (CATEGORIA_INGRESO_DEFAULT if monto >= 0 else CATEGORIA_EGRESO_DEFAULT), False


tipos_tx = mov["tipo_tx"] if "tipo_tx" in mov.columns else pd.Series("", index=mov.index)
clasificacion = [clasificar(c, t, m) for c, t, m in zip(mov["concepto"], tipos_tx, mov["monto"])]
mov["categoria"] = [c for c, _ in clasificacion]
mov["es_tesoreria"] = np.array([t for _, t in clasificacion], dtype=bool)
mov["flujo"] = np.where(mov["es_tesoreria"], "Tesorería", "Operativo")

tabla_categorias = (
    mov.groupby(["flujo", "categoria"])
    .agg(movimientos=("monto", "size"),
         ingresos=("monto", lambda s: s[s > 0].sum() / 1e6),
         egresos=("monto", lambda s: s[s < 0].sum() / 1e6),
         neto=("monto", lambda s: s.sum() / 1e6))
    .rename(columns={"ingresos": "ingresos (M MXN)", "egresos": "egresos (M MXN)", "neto": "neto (M MXN)"})
)
display(tabla_categorias.style.format({"movimientos": "{:,.0f}", "ingresos (M MXN)": "{:,.3f}",
                                       "egresos (M MXN)": "{:,.3f}", "neto (M MXN)": "{:,.3f}"}))

# Diagnóstico: ¿alguna cuenta cierra en cero todos los días? (cuenta concentradora / saldo cero)
neto_banco_dia = mov.groupby(["fecha", "banco"])["monto"].sum().unstack("banco")
for banco in neto_banco_dia.columns:
    if (neto_banco_dia[banco].abs() < 1).all():
        print(f"ℹ️ {banco}: el neto diario es 0 todos los días → sus movimientos de tesorería (barridos y giros) "
              "compensan exactamente la operación (cuenta de saldo cero). El objetivo útil es el flujo operativo.")

### 4.5 Construcción de las series diarias
Series en días hábiles; los días sin movimientos valen 0 y los movimientos de fin de semana pasan al siguiente día hábil. Se generan:
- `OPERATIVO_TOTAL`: flujo operativo de todos los bancos (**objetivo principal**).
- `OPERATIVO_<banco>`: flujo operativo por banco.
- `NETO_TOTAL`: flujo neto incluyendo tesorería (lo que se ve en el saldo consolidado).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9.5,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#b5b4ae",
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e5e4e0", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 10.5, "axes.titleweight": "bold",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "legend.frameon": False,
})
COLOR_REAL, COLOR_NEUTRO = "#0b0b0b", "#b5b4ae"
COLOR_POSITIVO, COLOR_NEGATIVO = "#2a78d6", "#e34948"
# Paleta categórica en orden fijo (validada para daltonismo); cada modelo conserva siempre su color y marcador
COLORES_MODELO = {
    "Chronos-2": "#2a78d6", "Chronos-2 + temporada": "#4a3aa7", "Chronos-2 (cross-learning)": "#eb6834",
    "Chronos-Bolt Small": "#1baf7a", "ETS": "#eda100", "SARIMAX": "#e87ba4", "SARIMAX + temporada": "#e34948",
    "Naive (referencia)": "#008300",
}
MARCADORES = {"Chronos-2": "o", "Chronos-2 + temporada": "P", "Chronos-2 (cross-learning)": "s",
              "Chronos-Bolt Small": "D", "ETS": "^", "SARIMAX": "v", "SARIMAX + temporada": "*",
              "Naive (referencia)": "X"}
EN_MILLONES = mticker.FuncFormatter(lambda v, _: f"{v / 1e6:,.1f}")

# Fecha de la serie: los fines de semana pasan al siguiente día hábil
if FRECUENCIA.upper() == "B":
    mov["fecha_serie"] = mov["fecha"].map(pd.offsets.BDay().rollforward).astype("datetime64[ns]")
    n_fds = int((mov["fecha"] != mov["fecha_serie"]).sum())
    if n_fds:
        print(f"ℹ️ {n_fds} movimientos de fin de semana se asignaron al siguiente día hábil.")
else:
    mov["fecha_serie"] = mov["fecha"]

fechas = pd.date_range(mov["fecha_serie"].min(), mov["fecha_serie"].max(), freq=FRECUENCIA).astype("datetime64[ns]")


def construir_series(m):
    definiciones = {"OPERATIVO_TOTAL": m[~m["es_tesoreria"]]}
    if SERIES_POR_BANCO:
        for banco in sorted(m["banco"].unique()):
            definiciones[f"OPERATIVO_{banco}"] = m[~m["es_tesoreria"] & (m["banco"] == banco)]
    definiciones["NETO_TOTAL"] = m
    partes = []
    for nombre, sub in definiciones.items():
        diario = sub.groupby("fecha_serie")["monto"].sum().reindex(fechas, fill_value=0.0)
        partes.append(pd.DataFrame({"unique_id": nombre, "ds": fechas, "y": diario.to_numpy(dtype=float)}))
    return pd.concat(partes, ignore_index=True)


series = construir_series(mov)
IDS = list(series["unique_id"].unique())
N_SERIES = len(IDS)
pivote = series.pivot(index="ds", columns="unique_id", values="y")[IDS]

n_dias = len(fechas)
print(f"Historia: {n_dias} días hábiles ({fechas[0]:%d-%m-%Y} → {fechas[-1]:%d-%m-%Y}) × {N_SERIES} series")
if n_dias < 60:
    print(f"⚠️ Con {n_dias} días la evaluación sirve para validar el pipeline de punta a punta, no para elegir "
          "un modelo con significancia. Para eso exporta al menos 12 meses de movimientos.")

print("\nFlujo diario (millones MXN):")
display((pivote / 1e6).rename_axis(None, axis=1).style.format("{:,.3f}").format_index("{:%a %d-%m-%Y}"))

# Gráfica: un panel por serie, barras con color según el signo del flujo
ncols = 2
nrows = int(np.ceil(N_SERIES / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(11, 2.9 * nrows), squeeze=False)
posiciones = np.arange(n_dias)
paso = max(1, n_dias // 8)
for ax, uid in zip(axes.flat, IDS):
    valores = pivote[uid].to_numpy()
    ax.bar(posiciones, valores, width=0.7, color=np.where(valores >= 0, COLOR_POSITIVO, COLOR_NEGATIVO))
    ax.axhline(0, color=COLOR_NEUTRO, lw=1)
    ax.set_title(uid, loc="left")
    ax.yaxis.set_major_formatter(EN_MILLONES)
    ax.set_xticks(posiciones[::paso], [f"{f:%d-%b}" for f in fechas[::paso]])
for ax in axes.flat[N_SERIES:]:
    ax.set_visible(False)
fig.supylabel("Flujo neto diario (millones MXN)", color="#52514e")
fig.legend(handles=[Patch(color=COLOR_POSITIVO, label="Entrada neta"), Patch(color=COLOR_NEGATIVO, label="Salida neta")],
           loc="upper right", ncol=2)
fig.suptitle("Series diarias a pronosticar", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

### 4.6 Carga de los modelos de Hugging Face
`BaseChronosPipeline.from_pretrained` lee la configuración del repositorio y devuelve la clase correcta (`Chronos2Pipeline` o `ChronosBoltPipeline`). Se mide el tiempo de carga (la primera vez incluye la descarga) y la memoria de GPU que ocupan los pesos.

In [ ]:
from chronos import BaseChronosPipeline

pipelines, TIEMPOS_CARGA, filas_modelos = {}, {}, []
for nombre, repo in REPOS_HF.items():
    memoria_antes = 0
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        memoria_antes = torch.cuda.memory_allocated()
    t0 = time.perf_counter()
    pipe = BaseChronosPipeline.from_pretrained(repo, device_map=DEVICE, dtype=TORCH_DTYPE, token=HF_TOKEN)
    sincronizar()
    TIEMPOS_CARGA[nombre] = time.perf_counter() - t0
    pipelines[nombre] = pipe
    primer_param = next(pipe.model.parameters())
    filas_modelos.append({
        "modelo": nombre,
        "repositorio": repo,
        "clase": type(pipe).__name__,
        "parámetros (M)": sum(p.numel() for p in pipe.model.parameters()) / 1e6,
        "contexto máx. (pasos)": pipe.model_context_length,
        "horizonte nativo (pasos)": pipe.model_prediction_length,
        "cuantiles entrenados": len(pipe.quantiles),
        "dispositivo": str(primer_param.device),
        "dtype": str(primer_param.dtype).replace("torch.", ""),
        "carga (s)": TIEMPOS_CARGA[nombre],
        "VRAM pesos (MB)": (torch.cuda.memory_allocated() - memoria_antes) / 1024**2 if DEVICE == "cuda" else np.nan,
    })
    print(f"✅ {nombre} cargado en {TIEMPOS_CARGA[nombre]:.1f} s")

tabla_modelos = pd.DataFrame(filas_modelos).set_index("modelo")
display(tabla_modelos.style.format({"parámetros (M)": "{:,.1f}", "carga (s)": "{:,.1f}",
                                    "VRAM pesos (MB)": "{:,.0f}"}, na_rep="—"))

## 5. Pipeline de inferencia
Todos los modelos comparten la misma interfaz: reciben la historia en formato largo (`unique_id`, `ds`, `y`) y devuelven, por serie y fecha futura, el pronóstico puntual (mediana) y los cuantiles P10/P50/P90.

- **Chronos-2 / Chronos-Bolt** usan `predict_df`, que genera los cuantiles directamente (no hay muestreo aleatorio: el resultado es determinista).
- **ETS / SARIMAX** convierten su intervalo gaussiano en cuantiles. Con historia corta reducen su especificación automáticamente: ETS agrega tendencia amortiguada con ≥ 10 días y estacionalidad semanal con ≥ 12; SARIMAX usa solo media con < 8 días, AR(1) con ≥ 8, estacionalidad semanal con ≥ 15 y regresores de calendario con ≥ 30. Si el ajuste falla o diverge, se usa el respaldo naive y se reporta.
- **Covariable `temporada`**: dos indicadores conocidos a futuro, `temp_ing_alto` (enero a marzo) y `temp_gasto_alto` (septiembre a diciembre).
  - *Chronos-2 + temporada* la recibe en el contexto y en `future_df`.
  - *SARIMAX + temporada* la recibe como columnas adicionales de `exog`.

  La covariable solo se usa en un origen cuando se cumplen dos condiciones: la historia ya contiene al menos dos temporadas con `MIN_DIAS_TEMPORADA` días hábiles o más cada una, y todas las temporadas del horizonte ya aparecieron en ella. Si no se cumplen, el modelo se comporta igual que sin covariable: no se puede estimar el efecto de una temporada nunca vista. La temporada base ("normal", en cuanto se observa) no lleva indicador, para no duplicar la constante de SARIMAX. Las versiones sin covariable reciben exactamente los mismos datos, menos esas columnas.

### 5.1 Adaptadores de cada modelo

In [ ]:
from scipy.stats import norm
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

COLS_Q = {q: f"P{int(round(q * 100))}" for q in CUANTILES}
COL_PUNTO = "pronostico"
COLUMNAS_SALIDA = ["unique_id", "ds", COL_PUNTO, *COLS_Q.values(), "respaldo"]


def fechas_futuras(ultima_fecha, h):
    return pd.date_range(ultima_fecha, periods=h + 1, freq=FRECUENCIA)[1:].astype("datetime64[ns]")


# ---------- Covariable temporada (conocida a futuro) ----------
COVARIABLES_TEMPORADA = ["temp_ing_alto", "temp_gasto_alto"]


def covariables_temporada(fechas_idx):
    """Indicadores 0/1 de temporada para cada fecha (float: Chronos-2 trata los booleanos como categorías)."""
    f = pd.DatetimeIndex(fechas_idx)
    return pd.DataFrame({
        "temp_ing_alto": f.month.isin(MESES_INGRESO_ALTO).astype(float),
        "temp_gasto_alto": f.month.isin(MESES_GASTO_ALTO).astype(float),
    })


def etiquetas_temporada(fechas_idx):
    f = pd.DatetimeIndex(fechas_idx)
    return np.where(f.month.isin(MESES_INGRESO_ALTO), "ingreso alto",
                    np.where(f.month.isin(MESES_GASTO_ALTO), "gasto alto", "normal"))


def _covariables_utiles(fechas_hist, fechas_sig):
    """Indicadores de temporada que se pueden usar en este origen (lista vacía = modelo sin covariable).

    - La historia debe contener al menos dos temporadas, cada una con MIN_DIAS_TEMPORADA días hábiles o más:
      si no, no hay con qué estimar su efecto.
    - Todas las temporadas del horizonte deben estar ya en la historia: no se extrapola a una temporada
      nunca vista (p. ej. el primer abril "normal" después de haber visto solo sep–mar).
    - Se omite el indicador de la temporada base ("normal" si ya se observó) para no duplicar la constante.
    """
    conteo = pd.Series(etiquetas_temporada(fechas_hist)).value_counts()
    vistas = set(conteo[conteo >= MIN_DIAS_TEMPORADA].index)
    if len(vistas) < 2 or not set(etiquetas_temporada(fechas_sig)) <= vistas:
        return []
    base = "normal" if "normal" in vistas else "gasto alto"
    return [col for etiqueta, col in (("ingreso alto", "temp_ing_alto"), ("gasto alto", "temp_gasto_alto"))
            if etiqueta in vistas and etiqueta != base]


# ---------- Modelos fundacionales (Hugging Face) ----------
def pronosticar_chronos(pipe, contexto, h, con_temporada=False, **opciones):
    df = contexto[["unique_id", "ds", "y"]].sort_values(["unique_id", "ds"], kind="stable").reset_index(drop=True)
    if con_temporada:
        fechas_ctx = pd.DatetimeIndex(df["ds"].drop_duplicates().sort_values())
        utiles = _covariables_utiles(fechas_ctx, fechas_futuras(fechas_ctx[-1], h))
        if utiles:
            df = pd.concat([df, covariables_temporada(df["ds"])[utiles]], axis=1)
            ultimas = df.groupby("unique_id", sort=False)["ds"].max()
            futuro = pd.concat([pd.DataFrame({"unique_id": uid, "ds": fechas_futuras(ultima, h)})
                                for uid, ultima in ultimas.items()], ignore_index=True)
            futuro = pd.concat([futuro, covariables_temporada(futuro["ds"])[utiles]], axis=1)
            opciones["future_df"] = futuro
    pred = pipe.predict_df(
        df, id_column="unique_id", timestamp_column="ds", target="y",
        prediction_length=h, quantile_levels=CUANTILES, freq=FRECUENCIA, **opciones,
    )
    pred = pred.rename(columns={"predictions": COL_PUNTO, **{str(q): c for q, c in COLS_Q.items()}})
    pred["ds"] = pd.to_datetime(pred["ds"]).astype("datetime64[ns]")
    pred["respaldo"] = False
    return pred[COLUMNAS_SALIDA]


# ---------- Modelos estadísticos (statsmodels) ----------
def calendario(fechas_idx):
    """Regresores exógenos de calendario para SARIMAX."""
    f = pd.DatetimeIndex(fechas_idx)
    X = pd.DataFrame(index=range(len(f)))
    for d, nombre in enumerate(["lun", "mar", "mie", "jue"]):     # viernes = categoría base
        X[nombre] = (f.dayofweek == d).astype(float)
    X["quincena"] = ((f.day >= 13) & (f.day <= 16)).astype(float)
    X["fin_mes"] = ((f + pd.offsets.BDay(2)).month != f.month).astype(float)
    return X


def _escala(y):
    e = float(np.mean(np.abs(y)))
    return e if e > 0 else 1.0


def _naive_una(y, fechas_hist, h, uid=None):
    dif = np.diff(y)
    sd1 = float(np.std(dif, ddof=1)) if len(dif) >= 2 else (float(np.abs(dif).mean()) if len(dif) else 0.0)
    return np.full(h, y[-1], dtype=float), sd1 * np.sqrt(np.arange(1, h + 1))


def _ets_una(y, fechas_hist, h, uid=None):
    n, escala = len(y), _escala(y)
    ys = pd.Series(y / escala)
    if np.ptp(ys) == 0:
        return np.full(h, y[-1], dtype=float), np.zeros(h)
    estacional, tendencia = n >= 2 * ESTACIONALIDAD + 2, n >= 10
    res = ETSModel(ys, error="add", trend="add" if tendencia else None, damped_trend=tendencia,
                   seasonal="add" if estacional else None,
                   seasonal_periods=ESTACIONALIDAD if estacional else None).fit(disp=False, maxiter=2000)
    pred = res.get_prediction(start=n, end=n + h - 1)
    return (np.asarray(pred.predicted_mean, float) * escala,
            np.sqrt(np.asarray(pred.var_pred_mean, float)) * escala)


def especificacion_sarimax(n):
    if n >= 3 * ESTACIONALIDAD:
        return (1, 0, 1), (1, 0, 0, ESTACIONALIDAD)
    if n >= 8:
        return (1, 0, 0), (0, 0, 0, 0)
    return (0, 0, 0), (0, 0, 0, 0)            # historia mínima: media + ruido


_cache_sarimax = None   # solo se activa durante el backtest (ver SARIMAX_REESTIMAR_CADA)


def _sarimax_una(y, fechas_hist, h, uid=None, con_temporada=False):
    n, escala = len(y), _escala(y)
    ys = pd.Series(y / escala)
    if np.ptp(ys) == 0:
        return np.full(h, y[-1], dtype=float), np.zeros(h)
    orden, orden_est = especificacion_sarimax(n)
    fechas_sig = fechas_futuras(fechas_hist[-1], h)
    partes_hist, partes_sig = [], []
    if USAR_EXOGENAS_SARIMAX and n >= 30:
        partes_hist.append(calendario(fechas_hist))
        partes_sig.append(calendario(fechas_sig))
    if con_temporada:
        utiles = _covariables_utiles(fechas_hist, fechas_sig)
        partes_hist.append(covariables_temporada(fechas_hist)[utiles])
        partes_sig.append(covariables_temporada(fechas_sig)[utiles])
    X = X_fut = None
    columnas = ()
    if partes_hist:
        X_hist, X_sig = pd.concat(partes_hist, axis=1), pd.concat(partes_sig, axis=1)
        columnas = tuple(X_hist.columns[X_hist.std() > 0])
        if columnas:
            X, X_fut = X_hist[list(columnas)].to_numpy(), X_sig[list(columnas)].to_numpy()
    modelo = SARIMAX(ys, exog=X, order=orden, seasonal_order=orden_est, trend="c")

    # En el backtest: re-estimación cada SARIMAX_REESTIMAR_CADA orígenes, arrancando desde los parámetros
    # anteriores; entre re-estimaciones solo se filtra con esos parámetros (sin usar datos del futuro).
    clave = (uid, con_temporada, orden, orden_est, columnas)
    previo = _cache_sarimax.get(clave) if (_cache_sarimax is not None and uid is not None) else None
    if previo is not None and len(previo["params"]) != len(modelo.start_params):
        previo = None
    if previo is not None and previo["usos"] < SARIMAX_REESTIMAR_CADA - 1:
        res = modelo.filter(previo["params"])
        previo["usos"] += 1
    else:
        res = modelo.fit(disp=False, maxiter=500, start_params=None if previo is None else previo["params"])
        if _cache_sarimax is not None and uid is not None:
            _cache_sarimax[clave] = {"params": np.asarray(res.params), "usos": 0}
    fc = res.get_forecast(h, exog=X_fut)
    return np.asarray(fc.predicted_mean, float) * escala, np.asarray(fc.se_mean, float) * escala


def _es_razonable(mu, sd, y):
    limite = 10 * max(float(np.abs(y).max()), 1.0)
    if not (np.all(np.isfinite(mu)) and np.all(np.isfinite(sd)) and np.all(np.abs(mu) <= limite)):
        return False
    return np.ptp(y) == 0 or bool(np.all(sd >= 1e-3 * np.std(y)))   # descarta intervalos colapsados


def _pronostico_estadistico(contexto, h, ajustar_una):
    salidas = []
    for uid, g in contexto.sort_values("ds", kind="stable").groupby("unique_id", sort=False):
        y = g["y"].to_numpy(dtype=float)
        fechas_hist = pd.DatetimeIndex(g["ds"])
        respaldo = False
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                mu, sd = ajustar_una(y, fechas_hist, h, uid=uid)
            if not _es_razonable(mu, sd, y):
                raise ValueError("ajuste degenerado")
        except Exception:
            mu, sd = _naive_una(y, fechas_hist, h)
            respaldo = ajustar_una is not _naive_una
        bloque = pd.DataFrame({"unique_id": uid, "ds": fechas_futuras(fechas_hist[-1], h), COL_PUNTO: mu})
        for q, col in COLS_Q.items():
            bloque[col] = mu + norm.ppf(q) * sd
        bloque["respaldo"] = respaldo
        salidas.append(bloque)
    return pd.concat(salidas, ignore_index=True)[COLUMNAS_SALIDA]


# ---------- Registro de modelos: todos con la firma (contexto, h) -> DataFrame ----------
def _sarimax_temporada(y, fechas_hist, h, uid=None):
    return _sarimax_una(y, fechas_hist, h, uid=uid, con_temporada=True)


MODELOS = {
    "Chronos-2": lambda ctx, h: pronosticar_chronos(pipelines["Chronos-2"], ctx, h),
    "Chronos-2 + temporada": lambda ctx, h: pronosticar_chronos(pipelines["Chronos-2"], ctx, h, con_temporada=True),
    "Chronos-2 (cross-learning)": lambda ctx, h: pronosticar_chronos(pipelines["Chronos-2"], ctx, h, cross_learning=True),
    "Chronos-Bolt Small": lambda ctx, h: pronosticar_chronos(pipelines["Chronos-Bolt Small"], ctx, h),
    "ETS": lambda ctx, h: _pronostico_estadistico(ctx, h, _ets_una),
    "SARIMAX": lambda ctx, h: _pronostico_estadistico(ctx, h, _sarimax_una),
    "SARIMAX + temporada": lambda ctx, h: _pronostico_estadistico(ctx, h, _sarimax_temporada),
    "Naive (referencia)": lambda ctx, h: _pronostico_estadistico(ctx, h, _naive_una),
}
FAMILIA = {"Chronos-2": "Fundacional HF", "Chronos-2 + temporada": "Fundacional HF",
           "Chronos-2 (cross-learning)": "Fundacional HF", "Chronos-Bolt Small": "Fundacional HF",
           "ETS": "Estadístico", "SARIMAX": "Estadístico", "SARIMAX + temporada": "Estadístico",
           "Naive (referencia)": "Referencia"}
MODELO_BASE = {"Chronos-2 (cross-learning)": "Chronos-2", "Chronos-2 + temporada": "Chronos-2"}  # comparten pesos
PARES_TEMPORADA = {"Chronos-2": "Chronos-2 + temporada", "SARIMAX": "SARIMAX + temporada"}     # sin → con covariable

print("Modelos registrados:", ", ".join(MODELOS))
fechas_todas = pd.DatetimeIndex(fechas)
print("Covariable temporada en la historia:",
      covariables_temporada(fechas_todas).set_index(fechas_todas).resample("MS").max()
      .rename(index=lambda f: f.strftime("%b-%y")).astype(int).T.to_string())

In [ ]:
# Prueba rápida: pronóstico del siguiente día hábil con toda la historia
prueba = pd.concat({nombre: fn(series, 1) for nombre, fn in MODELOS.items()}, names=["modelo"]).reset_index(level=0)
tabla_prueba = prueba.pivot(index="unique_id", columns="modelo", values=COL_PUNTO)[list(MODELOS)].loc[IDS] / 1e6
print(f"Pronóstico puntual (mediana) para el {fechas_futuras(fechas[-1], 1)[0]:%d-%m-%Y}, millones MXN:")
display(tabla_prueba.rename_axis(None, axis=1).style.format("{:,.3f}"))

### 5.2 Backtest *walk-forward*
Para cada origen *t* se usa **solo** la historia hasta *t* y se pronostican los siguientes `HORIZONTE` días hábiles; luego el origen avanza un día. Así se evalúa igual que se usaría en producción, sin fuga de información del futuro. La primera llamada de cada modelo se ejecuta aparte (calentamiento de GPU y cachés) y se guarda como latencia en frío.

- **Periodo:** cada pronóstico se etiqueta por la fecha que pronostica: *antes* o *desde* `FECHA_CORTE` (01-09-2026, inicio de la temporada de mayor gasto). La sección 6.3 compara ambos periodos.
- **SARIMAX:** en el backtest re-estima sus parámetros cada `SARIMAX_REESTIMAR_CADA` orígenes, partiendo de la estimación anterior. Entre re-estimaciones filtra la historia nueva con los últimos parámetros. Nunca usa datos posteriores al origen, y reduce varias veces el tiempo de cómputo.

In [ ]:
fechas_serie = pd.DatetimeIndex(np.sort(series["ds"].unique()))
cortes = list(range(MIN_CONTEXTO, len(fechas_serie) - HORIZONTE + 1))[-MAX_ORIGENES:]
if not cortes:
    raise ValueError(f"Con {len(fechas_serie)} días no hay orígenes con MIN_CONTEXTO={MIN_CONTEXTO} "
                     f"y HORIZONTE={HORIZONTE}. Reduce alguno de los dos en 4.1.")

print(f"Orígenes: {len(cortes)} ({fechas_serie[cortes[0] - 1]:%d-%m} → {fechas_serie[cortes[-1] - 1]:%d-%m}) · "
      f"{N_SERIES} series · horizonte {HORIZONTE} → {len(cortes) * N_SERIES * HORIZONTE} pronósticos por modelo\n")

CORTE = pd.Timestamp(FECHA_CORTE)
ETQ_ANTES, ETQ_DESDE = f"Antes del {CORTE:%d-%m-%Y}", f"Desde el {CORTE:%d-%m-%Y}"

registros, latencias, LATENCIA_FRIA = [], [], {}
_cache_sarimax = {}                      # re-estimación periódica de SARIMAX solo dentro del backtest
for nombre, pronosticar in MODELOS.items():
    t_modelo = time.perf_counter()

    # Calentamiento: primera llamada (en frío), fuera de las estadísticas del backtest
    sincronizar(); t0 = time.perf_counter()
    pronosticar(series[series["ds"] < fechas_serie[cortes[0]]], HORIZONTE)
    sincronizar(); LATENCIA_FRIA[nombre] = time.perf_counter() - t0

    for i, k in enumerate(cortes, start=1):
        if i % 50 == 0:
            print(f"  {nombre:28s} origen {i}/{len(cortes)}", end="\r")
        contexto = series[series["ds"] < fechas_serie[k]]
        objetivo = series[series["ds"].isin(fechas_serie[k:k + HORIZONTE])]

        sincronizar(); t0 = time.perf_counter()
        pred = pronosticar(contexto, HORIZONTE)
        sincronizar(); dt = time.perf_counter() - t0

        referencia = (contexto.sort_values("ds", kind="stable").groupby("unique_id")
                      .agg(ultimo_obs=("y", "last"), mediana_ctx=("y", "median")))
        pred = (pred.merge(objetivo, on=["unique_id", "ds"], how="left")
                    .merge(referencia, left_on="unique_id", right_index=True))
        pred["h"] = pred.groupby("unique_id").cumcount() + 1
        pred["origen"] = fechas_serie[k - 1]
        pred["modelo"] = nombre
        registros.append(pred)
        latencias.append({"modelo": nombre, "origen": fechas_serie[k - 1], "segundos": dt})

    print(f"  {nombre:28s} {time.perf_counter() - t_modelo:7.1f} s   (1.ª llamada en frío: {LATENCIA_FRIA[nombre] * 1000:,.0f} ms)")
_cache_sarimax = None

resultados = pd.concat(registros, ignore_index=True)
latencias_backtest = pd.DataFrame(latencias)
assert resultados["y"].notna().all(), "Hay pronósticos sin valor real: revisa FRECUENCIA y las fechas."
resultados["periodo"] = np.where(resultados["ds"] < CORTE, ETQ_ANTES, ETQ_DESDE)
print("\nPronósticos por periodo (por modelo):",
      (resultados.groupby("periodo").size() // len(MODELOS)).reindex([ETQ_ANTES, ETQ_DESDE]).fillna(0).astype(int).to_dict())

respaldos = resultados.groupby("modelo", sort=False)["respaldo"].mean().mul(100)
if respaldos.gt(0).any():
    print("\nPronósticos que usaron el respaldo naive (%):", respaldos[respaldos > 0].round(1).to_dict())
display(resultados.head(8))

### 5.3 Pronóstico de los próximos días hábiles
Con toda la historia disponible, cada modelo pronostica `HORIZONTE_FINAL` días hábiles hacia adelante. Este es el entregable operativo del predictor.

In [ ]:
siguientes = fechas_futuras(fechas[-1], HORIZONTE_FINAL)
pronostico_final = pd.concat({n: fn(series, HORIZONTE_FINAL) for n, fn in MODELOS.items()},
                             names=["modelo"]).reset_index(level=0)

q_lo, q_hi = COLS_Q[min(CUANTILES)], COLS_Q[max(CUANTILES)]
vista_final = pronostico_final.assign(
    fecha=lambda d: d["ds"].dt.strftime("%d-%m-%Y"),
    valor=lambda d: (d[COL_PUNTO] / 1e6).map("{:,.2f}".format) + "  [" + (d[q_lo] / 1e6).map("{:,.2f}".format)
                    + " – " + (d[q_hi] / 1e6).map("{:,.2f}".format) + "]",
)
tabla_final = (vista_final.pivot(index=["unique_id", "fecha"], columns="modelo", values="valor")
               .reindex(columns=list(MODELOS)).reindex(IDS, level="unique_id"))
print(f"Pronóstico {siguientes[0]:%d-%m-%Y} → {siguientes[-1]:%d-%m-%Y}  ·  millones MXN: mediana [{q_lo} – {q_hi}]")
display(tabla_final.rename_axis(columns=None))

## 6. Cálculo de métricas
Pronosticar flujo es un problema de **regresión**, y accuracy, precision, recall y F1 son métricas de **clasificación**. Para usarlas, cada pronóstico se convierte en la decisión binaria que tesorería tomaría con él y se compara contra lo que realmente pasó:

| Evento (clase positiva) | Pregunta de negocio | Regla |
|---|---|---|
| **Dirección** | ¿Entra más efectivo que en el último día conocido? | valor > último dato observado |
| **Flujo alto** | ¿Será un día de cobranza alta? | valor > mediana del historial |
| **Déficit** | ¿Habrá salida neta de efectivo? | valor < 0 |

- **Accuracy**: proporción de días clasificados correctamente.
- **Precision**: de los días que el modelo alertó, cuántos ocurrieron (controla falsas alarmas).
- **Recall**: de los días que ocurrieron, cuántos alertó (controla sorpresas).
- **F1**: media armónica de precision y recall. **n/d** = indefinido (no hubo positivos reales ni predichos).

Se complementan con métricas de regresión: **MAE** y **RMSE** (millones MXN), **sesgo** (promedio de pronóstico − real: positivo = sobreestima), **sMAPE**, **rMAE** (MAE dividido entre el del naive; < 1 significa mejor que "mañana = hoy"), **WQL** (pérdida de cuantiles ponderada: califica todo el rango P10–P90; menor es mejor) y **cobertura** del intervalo P10–P90 (lo ideal es ≈ 80 %).

### 6.1 Métricas de regresión

In [ ]:
ETQ_COBERTURA = f"Cobertura {q_lo}–{q_hi} (%)"


def metricas_regresion(d):
    error = d[COL_PUNTO] - d["y"]
    denom = d["y"].abs() + d[COL_PUNTO].abs()
    pinball = sum(np.maximum(q * (d["y"] - d[c]), (q - 1) * (d["y"] - d[c])).sum() for q, c in COLS_Q.items())
    return pd.Series({
        "MAE": error.abs().mean(),
        "Sesgo": error.mean(),                      # pronóstico − real: > 0 sobreestima, < 0 subestima
        "RMSE": np.sqrt((error ** 2).mean()),
        "sMAPE (%)": (2 * error.abs() / denom).where(denom > 0).mean() * 100,
        "WQL": 2 * pinball / len(COLS_Q) / d["y"].abs().sum(),
        ETQ_COBERTURA: ((d["y"] >= d[q_lo]) & (d["y"] <= d[q_hi])).mean() * 100,
        "Ancho intervalo": (d[q_hi] - d[q_lo]).mean(),
        "Respaldos (%)": d["respaldo"].mean() * 100,
    })


tabla_regresion = resultados.groupby("modelo", sort=False).apply(metricas_regresion, include_groups=False)
tabla_regresion.insert(1, "rMAE vs naive", tabla_regresion["MAE"] / tabla_regresion.loc["Naive (referencia)", "MAE"])

vista_reg = tabla_regresion.copy()
for c in ["MAE", "Sesgo", "RMSE", "Ancho intervalo"]:
    vista_reg[c] = vista_reg[c] / 1e6
vista_reg = vista_reg.rename(columns={"MAE": "MAE (M MXN)", "Sesgo": "Sesgo (M MXN)", "RMSE": "RMSE (M MXN)",
                                      "Ancho intervalo": "Ancho intervalo (M MXN)"})
RESALTE = "font-weight: bold; background-color: #cde2fb"
display(vista_reg.style.format("{:,.3f}", na_rep="n/d")
        .format({"sMAPE (%)": "{:,.1f}", ETQ_COBERTURA: "{:,.0f}", "Respaldos (%)": "{:,.0f}"})
        .highlight_min(subset=["MAE (M MXN)", "rMAE vs naive", "RMSE (M MXN)", "sMAPE (%)", "WQL"], props=RESALTE))

mae_por_serie = (resultados.groupby(["unique_id", "modelo"], sort=False)
                 .apply(lambda d: (d[COL_PUNTO] - d["y"]).abs().mean(), include_groups=False)
                 .unstack("modelo")[list(MODELOS)].loc[IDS] / 1e6)
print("MAE por serie (millones MXN):")
display(mae_por_serie.rename_axis(None, axis=1).style.format("{:,.3f}").highlight_min(axis=1, props=RESALTE))

### 6.2 Métricas de clasificación: accuracy, precision, recall y F1-score

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

EVENTOS = {
    "Dirección (sube vs. último dato)": lambda d, col: d[col] > d["ultimo_obs"],
    "Flujo alto (> mediana histórica)": lambda d, col: d[col] > d["mediana_ctx"],
    "Déficit (flujo neto < 0)": lambda d, col: d[col] < 0,
}
EVENTO_PRINCIPAL = next(iter(EVENTOS))


def metricas_clasificacion(d, regla):
    real = regla(d, "y").astype(int).to_numpy()
    pred = regla(d, COL_PUNTO).astype(int).to_numpy()
    vn, fp, fn, vp = confusion_matrix(real, pred, labels=[0, 1]).ravel()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {
            "accuracy": accuracy_score(real, pred),
            "precision": precision_score(real, pred, zero_division=np.nan),
            "recall": recall_score(real, pred, zero_division=np.nan),
            "F1": f1_score(real, pred, zero_division=np.nan),
            "VP": vp, "FP": fp, "FN": fn, "VN": vn, "n": len(real),
        }


tabla_clasificacion = pd.DataFrame([
    {"evento": evento, "modelo": modelo, **metricas_clasificacion(d, regla)}
    for evento, regla in EVENTOS.items()
    for modelo, d in resultados.groupby("modelo", sort=False)
]).set_index(["evento", "modelo"])

formato_clf = {"accuracy": "{:.2f}", "precision": "{:.2f}", "recall": "{:.2f}", "F1": "{:.2f}",
               "VP": "{:d}", "FP": "{:d}", "FN": "{:d}", "VN": "{:d}", "n": "{:d}"}
for evento in EVENTOS:
    display(Markdown(f"**{evento}**  ·  VP/FP/FN/VN = verdaderos/falsos positivos/negativos"))
    display(tabla_clasificacion.loc[evento].style.format(formato_clf, na_rep="n/d")
            .highlight_max(subset=["accuracy", "precision", "recall", "F1"], props=RESALTE))

print("F1-score por evento:")
display(tabla_clasificacion["F1"].unstack("evento")[list(EVENTOS)].loc[list(MODELOS)]
        .rename_axis(None, axis=1).style.format("{:.2f}", na_rep="n/d"))

### 6.3 Reacción al cambio de temporada: antes y después del 01-09-2026
Las mismas métricas se calculan por separado para los pronósticos de fechas **anteriores** a `FECHA_CORTE` y para los de fechas **desde** ese día, cuando empieza la temporada de mayor gasto. En la segunda tabla, cada modelo con covariable se compara contra su versión sin ella:

- Un **cambio de MAE negativo** significa que la covariable mejora el pronóstico.
- Un **sesgo cercano a cero desde el corte** significa que el modelo anticipó el cambio de nivel en lugar de seguir pronosticando el nivel anterior.

La tercera tabla muestra el MAE por mes de la fecha pronosticada, para ver la reacción en cada transición: entrada a ene–mar, regreso a lo normal en abril y entrada a la temporada de mayor gasto en septiembre. Con un año de historia, solo la última se puede anticipar con la covariable (ver 9.2).

In [ ]:
periodos = [p for p in (ETQ_ANTES, ETQ_DESDE) if p in set(resultados["periodo"])]


def metricas_periodo(d):
    m = metricas_regresion(d)
    real, pred = d["y"] > d["ultimo_obs"], d[COL_PUNTO] > d["ultimo_obs"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m["F1 dirección"] = f1_score(real, pred, zero_division=np.nan)
    m["n"] = len(d)
    return m


tabla_periodo = resultados.groupby(["periodo", "modelo"], sort=False).apply(metricas_periodo, include_groups=False)
mae_naive = tabla_periodo.xs("Naive (referencia)", level="modelo")["MAE"]
tabla_periodo["rMAE vs naive"] = (tabla_periodo["MAE"].to_numpy()
                                  / mae_naive.reindex(tabla_periodo.index.get_level_values("periodo")).to_numpy())

METRICAS_PERIODO = ["MAE", "rMAE vs naive", "Sesgo", "WQL", ETQ_COBERTURA, "F1 dirección"]
vista_periodo = (tabla_periodo[METRICAS_PERIODO].unstack("periodo")
                 .reindex(columns=pd.MultiIndex.from_product([METRICAS_PERIODO, periodos])).loc[list(MODELOS)])
for c in ["MAE", "Sesgo"]:
    vista_periodo[c] = vista_periodo[c] / 1e6
vista_periodo = vista_periodo.rename(columns={"MAE": "MAE (M MXN)", "Sesgo": "Sesgo (M MXN)"}, level=0)
n_periodo = (tabla_periodo["n"].unstack("periodo").iloc[0]).reindex(periodos).astype(int)
print("Pronósticos por modelo y periodo:", n_periodo.to_dict())
fmt = {c: ("{:,.0f}" if c[0] == ETQ_COBERTURA else "{:,.2f}" if c[0] == "F1 dirección" else "{:,.3f}")
       for c in vista_periodo.columns}
display(vista_periodo.style.format(fmt, na_rep="n/d")
        .highlight_min(subset=[c for c in vista_periodo.columns if c[0] in ("MAE (M MXN)", "rMAE vs naive", "WQL")],
                       props=RESALTE))

# ---- Efecto de la covariable temporada: cada modelo contra su versión sin covariable
filas_efecto = []
for base, con in PARES_TEMPORADA.items():
    for p in periodos:
        a, b = tabla_periodo.loc[(p, base)], tabla_periodo.loc[(p, con)]
        filas_efecto.append({
            "modelo": base, "periodo": p,
            "MAE sin (M MXN)": a["MAE"] / 1e6, "MAE con (M MXN)": b["MAE"] / 1e6,
            "Cambio de MAE (%)": (b["MAE"] / a["MAE"] - 1) * 100,
            "Sesgo sin (M MXN)": a["Sesgo"] / 1e6, "Sesgo con (M MXN)": b["Sesgo"] / 1e6,
            "WQL sin": a["WQL"], "WQL con": b["WQL"],
        })
efecto_temporada = pd.DataFrame(filas_efecto).set_index(["modelo", "periodo"])
display(Markdown("**Efecto de la covariable `temporada`** (negativo = la covariable reduce el error)"))
display(efecto_temporada.style.format("{:,.3f}").format({"Cambio de MAE (%)": "{:+,.1f}"})
        .background_gradient(subset=["Cambio de MAE (%)"], cmap="RdBu", vmin=-30, vmax=30))

# ---- MAE por mes de la fecha pronosticada
def temporada_de(mes):
    return ("Mayor ingreso" if mes.month in MESES_INGRESO_ALTO else
            "Mayor gasto" if mes.month in MESES_GASTO_ALTO else "Normal")


resultados["mes"] = resultados["ds"].dt.to_period("M")
mae_mensual = (resultados.assign(err=(resultados[COL_PUNTO] - resultados["y"]).abs())
               .pivot_table(index="mes", columns="modelo", values="err", aggfunc="mean")[list(MODELOS)] / 1e6)
sesgo_mensual = (resultados.assign(err=resultados[COL_PUNTO] - resultados["y"])
                 .pivot_table(index="mes", columns="modelo", values="err", aggfunc="mean")[list(MODELOS)] / 1e6)
vista_mensual = mae_mensual.copy()
vista_mensual.insert(0, "Temporada", [temporada_de(m) for m in vista_mensual.index])
vista_mensual.index = [m.strftime("%b-%y") for m in vista_mensual.index]
display(Markdown("**MAE mensual (M MXN) por mes de la fecha pronosticada**  ·  resaltado: mejor modelo del mes"))
display(vista_mensual.style.format("{:,.2f}", subset=list(MODELOS))
        .highlight_min(subset=list(MODELOS), axis=1, props=RESALTE))

## 7. Medición de latencia
Cada modelo pronostica todas las series con toda la historia hasta `N_REPETICIONES_LATENCIA` veces (los modelos que acumulan más de 30 s se detienen a las 5 repeticiones); se reportan mediana y percentil 95 (más robustos que el promedio). El cronómetro se sincroniza con la GPU antes y después de cada llamada. Se incluyen también el tiempo de carga del modelo, la primera llamada en frío y el pico de memoria de GPU durante la inferencia. ETS y SARIMAX siempre corren en CPU.

In [ ]:
filas_lat = []
for nombre, pronosticar in MODELOS.items():
    es_hf = FAMILIA[nombre] == "Fundacional HF"
    memoria_base = 0
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
        memoria_base = torch.cuda.memory_allocated()
    tiempos = []
    for _ in range(N_REPETICIONES_LATENCIA):
        sincronizar(); t0 = time.perf_counter()
        pronosticar(series, HORIZONTE)
        sincronizar(); tiempos.append((time.perf_counter() - t0) * 1000)
        if len(tiempos) >= 5 and sum(tiempos) > 30_000:      # modelos lentos: basta con ≥ 5 mediciones
            break
    t = np.array(tiempos)
    filas_lat.append({
        "modelo": nombre,
        "dispositivo": DEVICE if es_hf else "cpu",
        "repeticiones": len(t),
        "carga (s)": TIEMPOS_CARGA.get(MODELO_BASE.get(nombre, nombre), 0.0),
        "1.ª llamada en frío (ms)": LATENCIA_FRIA[nombre] * 1000,
        "mediana (ms)": np.median(t),
        "p95 (ms)": np.percentile(t, 95),
        "desv. est. (ms)": t.std(ddof=1),
        "ms por serie": np.median(t) / N_SERIES,
        "series por segundo": N_SERIES / (np.median(t) / 1000),
        "mediana en backtest (ms)*": latencias_backtest.loc[latencias_backtest["modelo"] == nombre, "segundos"].median() * 1000,
        "VRAM pico inferencia (MB)": ((torch.cuda.max_memory_allocated() - memoria_base) / 1024**2
                                      if DEVICE == "cuda" and es_hf else np.nan),
    })

tabla_latencia = pd.DataFrame(filas_lat).set_index("modelo")
print(f"Latencia por llamada · {N_SERIES} series · {len(fechas)} días de contexto · horizonte {HORIZONTE} · "
      f"hasta {N_REPETICIONES_LATENCIA} repeticiones")
print(f"* En el backtest SARIMAX re-estima cada {SARIMAX_REESTIMAR_CADA} orígenes; aquí siempre estima desde cero.")
display(tabla_latencia.style.format("{:,.1f}", na_rep="—")
        .format({"carga (s)": "{:,.2f}", "dispositivo": "{}", "repeticiones": "{:d}"})
        .highlight_min(subset=["mediana (ms)", "p95 (ms)", "ms por serie"], props=RESALTE))

### 7.1 Escalabilidad con el número de series
En producción se pronosticarían muchas cuentas o conceptos a la vez. Aquí se replican las series para medir cómo crece la latencia por llamada: los modelos fundacionales procesan el lote en paralelo en la GPU, mientras que ETS/SARIMAX ajustan un modelo por serie. Para no alargar la corrida, un tamaño de lote no se mide cuando la proyección lineal desde el tamaño anterior supera 60 s (en CPU suele pasarles a SARIMAX y SARIMAX + temporada con 64 series).

In [ ]:
def replicar_series(df, n):
    base = list(df["unique_id"].unique())
    return pd.concat([df[df["unique_id"] == base[i % len(base)]].assign(unique_id=f"{base[i % len(base)]}#{i}")
                      for i in range(n)], ignore_index=True)


LIMITE_ESCALA_S = 60          # no se mide un tamaño cuya proyección lineal desde el anterior supere este tiempo
filas_esc, ultimo_medido = [], {}
for n in TAMANOS_LOTE:
    lote = replicar_series(series, n)
    for nombre, pronosticar in MODELOS.items():
        if nombre in ultimo_medido:
            n_prev, ms_prev = ultimo_medido[nombre]
            if ms_prev / 1000 * n / n_prev > LIMITE_ESCALA_S:
                filas_esc.append({"series en el lote": n, "modelo": nombre, "mediana (ms)": np.nan})
                continue
        tiempos = []
        for _ in range(3):
            sincronizar(); t0 = time.perf_counter()
            pronosticar(lote, HORIZONTE)
            sincronizar(); tiempos.append((time.perf_counter() - t0) * 1000)
            if tiempos[-1] > 5_000:                              # llamadas de varios segundos: una medición basta
                break
        filas_esc.append({"series en el lote": n, "modelo": nombre, "mediana (ms)": np.median(tiempos)})
        ultimo_medido[nombre] = (n, np.median(tiempos))

tabla_escala = (pd.DataFrame(filas_esc)
                .pivot(index="series en el lote", columns="modelo", values="mediana (ms)")[list(MODELOS)])
print(f"Latencia mediana por llamada (ms) · — = no medido: la proyección lineal superaba {LIMITE_ESCALA_S} s")
display(tabla_escala.rename_axis(None, axis=1).style.format("{:,.1f}", na_rep="—"))

fig, ax = plt.subplots(figsize=(8, 4.2))
for nombre in MODELOS:
    medidos = tabla_escala[nombre].dropna()
    ax.plot(medidos.index, medidos, color=COLORES_MODELO[nombre], marker=MARCADORES[nombre],
            lw=2, ms=7, label=nombre)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(TAMANOS_LOTE, [str(n) for n in TAMANOS_LOTE])
ax.grid(True, which="major", axis="both")
ax.set_xlabel("Series pronosticadas en una llamada")
ax.set_ylabel(f"Latencia mediana (ms, {DEVICE} para Chronos)")
ax.set_title("Escalabilidad de la inferencia", loc="left")
ax.legend(loc="upper left", fontsize=8.5)
plt.tight_layout()
plt.show()

## 8. Tablas y resúmenes
### 8.1 Tabla resumen comparativa
Ordenada por MAE del backtest completo, con el MAE separado antes y desde `FECHA_CORTE`. Las métricas de clasificación corresponden al evento principal (*dirección*). Se resalta el mejor valor de cada columna.

In [ ]:
direccion = tabla_clasificacion.loc[EVENTO_PRINCIPAL]
flujo_alto = tabla_clasificacion.loc["Flujo alto (> mediana histórica)"]

resumen = pd.DataFrame({
    "Familia": pd.Series(FAMILIA),
    "MAE (M MXN)": tabla_regresion["MAE"] / 1e6,
    "rMAE vs naive": tabla_regresion["rMAE vs naive"],
    **{f"MAE {p.lower()} (M MXN)": tabla_periodo.xs(p, level="periodo")["MAE"] / 1e6 for p in periodos},
    "WQL": tabla_regresion["WQL"],
    ETQ_COBERTURA: tabla_regresion[ETQ_COBERTURA],
    "Accuracy": direccion["accuracy"],
    "Precision": direccion["precision"],
    "Recall": direccion["recall"],
    "F1": direccion["F1"],
    "F1 flujo alto": flujo_alto["F1"],
    "Latencia mediana (ms)": tabla_latencia["mediana (ms)"],
    "Carga (s)": tabla_latencia["carga (s)"],
}).loc[list(MODELOS)].sort_values("MAE (M MXN)")

COLS_MAE = ["MAE (M MXN)", "rMAE vs naive", *[f"MAE {p.lower()} (M MXN)" for p in periodos], "WQL"]
display(resumen.style
        .format("{:,.3f}", na_rep="n/d", subset=COLS_MAE)
        .format("{:,.0f}", subset=[ETQ_COBERTURA])
        .format("{:.2f}", na_rep="n/d", subset=["Accuracy", "Precision", "Recall", "F1", "F1 flujo alto"])
        .format("{:,.1f}", subset=["Latencia mediana (ms)", "Carga (s)"])
        .highlight_min(subset=[*COLS_MAE, "Latencia mediana (ms)"], props=RESALTE)
        .highlight_max(subset=["Accuracy", "Precision", "Recall", "F1", "F1 flujo alto"], props=RESALTE))

### 8.2 Backtest alrededor del cambio de temporada
Real contra pronóstico a un día de los modelos con y sin la covariable `temporada`, desde seis semanas antes de `FECHA_CORTE` hasta el final de la historia. La línea vertical marca el corte; las tablas de la sección 6 tienen a todos los modelos.

In [ ]:
MODELOS_GRAFICA = [m for par in PARES_TEMPORADA.items() for m in par]
r1 = resultados[resultados["h"] == 1]
desde = CORTE - pd.offsets.BDay(30) if fechas_serie[0] < CORTE <= fechas_serie[-1] else fechas_serie[-60]
ncols = 2
nrows = int(np.ceil(N_SERIES / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.4 * nrows), squeeze=False)
for ax, uid in zip(axes.flat, IDS):
    hist = series[(series["unique_id"] == uid) & (series["ds"] >= desde)]
    ax.axhline(0, color=COLOR_NEUTRO, lw=0.8)
    if fechas_serie[0] < CORTE <= fechas_serie[-1]:
        ax.axvline(CORTE, color=COLOR_REAL, lw=1, ls="--", zorder=1)
    ax.plot(hist["ds"], hist["y"], color=COLOR_REAL, lw=2.2, marker="o", ms=3.5, label="Real", zorder=5)
    for nombre in MODELOS_GRAFICA:
        d = r1[(r1["unique_id"] == uid) & (r1["modelo"] == nombre) & (r1["ds"] >= desde)]
        ax.plot(d["ds"], d[COL_PUNTO], color=COLORES_MODELO[nombre], marker=MARCADORES[nombre],
                lw=1.6, ms=5.5, markeredgecolor="white", markeredgewidth=0.6, label=nombre, zorder=4)
    ax.set_title(uid, loc="left")
    ax.yaxis.set_major_formatter(EN_MILLONES)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
for ax in axes.flat[N_SERIES:]:
    ax.set_visible(False)
fig.supylabel("Flujo neto diario (millones MXN)", color="#52514e")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, bbox_to_anchor=(0.5, 0.0))
fig.suptitle(f"Backtest walk-forward: real vs. pronóstico a 1 día (línea punteada = {CORTE:%d-%m-%Y})",
             x=0.01, ha="left", fontweight="bold")
plt.tight_layout(rect=(0, 0.06, 1, 0.96))
plt.show()

### 8.2.1 Sesgo mensual: ¿anticipa o detecta tarde el cambio?
Promedio de *pronóstico − real* por mes de la fecha pronosticada, considerando todas las series y horizontes, para los modelos con y sin covariable. Las bandas de color marcan las temporadas. Un sesgo que se dispara justo al entrar a una temporada y luego se corrige indica que el modelo detectó el cambio tarde. Si se mantiene cerca de cero, el modelo lo anticipó.

In [ ]:
inicio_mes = sesgo_mensual.index.to_timestamp()
meses_eje = inicio_mes + pd.Timedelta(days=14)                 # cada punto a mitad de su mes
fig, ax = plt.subplots(figsize=(11, 4.4))
for m in inicio_mes:
    if m.month in MESES_INGRESO_ALTO or m.month in MESES_GASTO_ALTO:
        ax.axvspan(m, m + pd.offsets.MonthBegin(1), zorder=0, lw=0,
                   color="#e6f0fb" if m.month in MESES_INGRESO_ALTO else "#fbe9e7")
ax.axhline(0, color=COLOR_REAL, lw=1)
for nombre in MODELOS_GRAFICA:
    ax.plot(meses_eje, sesgo_mensual[nombre], color=COLORES_MODELO[nombre], marker=MARCADORES[nombre],
            lw=2, ms=7, markeredgecolor="white", markeredgewidth=0.8, label=nombre)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+,.1f}"))   # ya está en millones
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=15))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%y"))
ax.set_xlim(inicio_mes[0], inicio_mes[-1] + pd.offsets.MonthBegin(1))
ax.set_ylabel("Pronóstico − real (millones MXN)")
ax.set_title("Sesgo mensual del pronóstico (azul claro = mayor ingreso · rojo claro = mayor gasto)", loc="left")
ax.legend(loc="lower right", ncol=2, fontsize=8.5)
plt.tight_layout()
plt.show()

print("Sesgo mensual (M MXN):")
display(sesgo_mensual[MODELOS_GRAFICA].rename(index=lambda m: m.strftime("%b-%y")).rename_axis(None, axis=1)
        .style.format("{:+,.2f}").background_gradient(cmap="RdBu_r", vmin=-2, vmax=2))

### 8.3 Exactitud vs. costo computacional
Dos gráficas separadas (cada una con su propia escala): error relativo al naive y latencia por llamada.

In [ ]:
orden = resumen.index.tolist()[::-1]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 0.55 * len(orden) + 1.6))

valores = resumen.loc[orden, "rMAE vs naive"]
ax1.barh(orden, valores, color=COLOR_POSITIVO, height=0.6)
ax1.axvline(1, color=COLOR_REAL, lw=1, ls="--")
ax1.text(1, len(orden) - 0.4, " naive = 1", color="#52514e", va="bottom", fontsize=8.5)
for i, v in enumerate(valores):
    ax1.text(v, i, f" {v:.2f}", va="center", color="#52514e", fontsize=8.5)
ax1.set_title("MAE relativo al naive (menor es mejor)", loc="left")
ax1.grid(True, axis="x"); ax1.grid(False, axis="y")
ax1.set_xlim(0, max(valores.max(), 1) * 1.18)

lat = resumen.loc[orden, "Latencia mediana (ms)"]
ax2.barh(orden, lat, color=COLOR_POSITIVO, height=0.6)
for i, v in enumerate(lat):
    ax2.text(v, i, f" {v:,.1f}", va="center", color="#52514e", fontsize=8.5)
ax2.set_xscale("log")
ax2.set_title(f"Latencia mediana por llamada, ms ({N_SERIES} series, escala log)", loc="left")
ax2.grid(True, axis="x"); ax2.grid(False, axis="y")
ax2.set_xlim(lat.min() / 2, lat.max() * 6)
ax2.tick_params(axis="y", labelleft=False)
plt.tight_layout()
plt.show()

### 8.4 Pronóstico del siguiente día hábil por modelo
Punto = mediana; barra = intervalo P10–P90.

In [ ]:
dia = siguientes[0]
siguiente = pronostico_final[pronostico_final["ds"] == dia]
ncols = 2
nrows = int(np.ceil(N_SERIES / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.6 * nrows + 0.6), squeeze=False)
nombres = list(MODELOS)[::-1]
for ax, uid in zip(axes.flat, IDS):
    d = siguiente[siguiente["unique_id"] == uid].set_index("modelo").loc[nombres]
    for i, nombre in enumerate(nombres):
        ax.plot([d.loc[nombre, q_lo], d.loc[nombre, q_hi]], [i, i], color=COLORES_MODELO[nombre], lw=2.5,
                solid_capstyle="round")
        ax.plot(d.loc[nombre, COL_PUNTO], i, marker=MARCADORES[nombre], color=COLORES_MODELO[nombre], ms=8,
                markeredgecolor="white", markeredgewidth=1)
    ultimo = series.loc[series["unique_id"] == uid, "y"].iloc[-1]
    ax.axvline(ultimo, color=COLOR_REAL, lw=1, ls="--")
    ax.axvline(0, color=COLOR_NEUTRO, lw=0.8)
    ax.set_yticks(range(len(nombres)), nombres)
    ax.set_title(uid, loc="left")
    ax.xaxis.set_major_formatter(EN_MILLONES)
    ax.grid(True, axis="x"); ax.grid(False, axis="y")
for ax in axes.flat[N_SERIES:]:
    ax.set_visible(False)
fig.supxlabel("Millones MXN  ·  línea punteada = último día observado", color="#52514e")
fig.suptitle(f"Pronóstico para el {dia:%d-%m-%Y}: mediana e intervalo {q_lo}–{q_hi}", x=0.01, ha="left",
             fontweight="bold")
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

### 8.5 Exportar resultados a Excel

In [ ]:
ARCHIVO_SALIDA = "resultados_predictor_flujo.xlsx"
with pd.ExcelWriter(ARCHIVO_SALIDA, engine="openpyxl") as xls:
    resumen.to_excel(xls, sheet_name="resumen")
    tabla_regresion.to_excel(xls, sheet_name="metricas_regresion")
    tabla_clasificacion.to_excel(xls, sheet_name="metricas_clasificacion")
    tabla_periodo.to_excel(xls, sheet_name="metricas_por_periodo")
    efecto_temporada.to_excel(xls, sheet_name="efecto_temporada")
    mae_mensual.rename(index=str).to_excel(xls, sheet_name="mae_mensual")
    sesgo_mensual.rename(index=str).to_excel(xls, sheet_name="sesgo_mensual")
    mae_por_serie.to_excel(xls, sheet_name="mae_por_serie")
    tabla_latencia.to_excel(xls, sheet_name="latencia")
    tabla_escala.to_excel(xls, sheet_name="escalabilidad")
    pronostico_final.to_excel(xls, sheet_name="pronostico", index=False)
    resultados.assign(mes=resultados["mes"].astype(str)).to_excel(xls, sheet_name="backtest", index=False)
    series.to_excel(xls, sheet_name="series_diarias", index=False)
    tabla_categorias.to_excel(xls, sheet_name="categorias")
    mov.to_excel(xls, sheet_name="movimientos_unicos", index=False)
print("✅ Resultados guardados en", ARCHIVO_SALIDA)

try:
    from google.colab import files
    files.download(ARCHIVO_SALIDA)
except ImportError:
    pass

## 9. Conclusiones parciales
### 9.1 Resultados de esta corrida
Se generan a partir de las tablas anteriores, así que se actualizan solos si cambias el archivo o los parámetros.

In [ ]:
def _f(v, patron="{:,.2f}"):
    return "n/d" if pd.isna(v) else patron.format(v)


n_eval = len(resultados) // len(MODELOS)
cobertura_ideal = (max(CUANTILES) - min(CUANTILES)) * 100
mejor_mae = resumen["MAE (M MXN)"].idxmin()
mejor_wql = resumen["WQL"].idxmin()
mejor_cob = (resumen[ETQ_COBERTURA] - cobertura_ideal).abs().idxmin()
mejor_f1 = resumen["F1"].idxmax() if resumen["F1"].notna().any() else None
superan_naive = resumen.index[resumen["rMAE vs naive"] < 1].tolist()
hf = resumen[resumen["Familia"] == "Fundacional HF"]
est = resumen[resumen["Familia"] == "Estadístico"]
hf_rapido = hf["Latencia mediana (ms)"].idxmin()

lineas = [
    f"- **Base de evaluación:** {n_eval} pronósticos por modelo ({len(cortes)} orígenes × {N_SERIES} series × "
    f"{HORIZONTE} paso(s)), sobre {len(fechas)} días hábiles de historia.",
    f"- **Menor error puntual:** {mejor_mae}, con MAE de {_f(resumen.loc[mejor_mae, 'MAE (M MXN)'])} M MXN "
    f"(rMAE {_f(resumen.loc[mejor_mae, 'rMAE vs naive'])}).",
    f"- **Mejor pronóstico probabilístico (WQL):** {mejor_wql} (WQL {_f(resumen.loc[mejor_wql, 'WQL'], '{:.3f}')}).",
    f"- **Superan al naive (rMAE < 1):** {', '.join(superan_naive) if superan_naive else 'ningún modelo'}.",
    (f"- **Dirección del flujo:** mejor F1 para {mejor_f1} ({_f(resumen.loc[mejor_f1, 'F1'])}; accuracy "
     f"{_f(resumen.loc[mejor_f1, 'Accuracy'])}, precision {_f(resumen.loc[mejor_f1, 'Precision'])}, "
     f"recall {_f(resumen.loc[mejor_f1, 'Recall'])})." if mejor_f1 else
     "- **Dirección del flujo:** F1 indefinido para todos los modelos en este periodo."),
    f"- **Calibración del intervalo:** el más cercano al {cobertura_ideal:.0f} % ideal es {mejor_cob} "
    f"({_f(resumen.loc[mejor_cob, ETQ_COBERTURA], '{:.0f}')} %).",
    f"- **Latencia:** el fundacional más rápido es {hf_rapido} ({_f(hf.loc[hf_rapido, 'Latencia mediana (ms)'], '{:,.1f}')} ms "
    f"por llamada en {DEVICE}); ETS/SARIMAX tardan {_f(est['Latencia mediana (ms)'].min(), '{:,.1f}')}–"
    f"{_f(est['Latencia mediana (ms)'].max(), '{:,.1f}')} ms en CPU, y su costo crece linealmente con el número de series.",
]
if ETQ_DESDE in periodos:
    col_desde = f"MAE {ETQ_DESDE.lower()} (M MXN)"
    mejor_desde = resumen[col_desde].idxmin()
    lineas.insert(2, f"- **{ETQ_DESDE} (temporada de mayor gasto):** menor MAE para {mejor_desde} "
                     f"({_f(resumen.loc[mejor_desde, col_desde])} M MXN).")
for base, con in PARES_TEMPORADA.items():
    partes = [f"{p.lower()}: MAE {efecto_temporada.loc[(base, p), 'Cambio de MAE (%)']:+.1f} %, sesgo "
              f"{efecto_temporada.loc[(base, p), 'Sesgo sin (M MXN)']:+.2f} → "
              f"{efecto_temporada.loc[(base, p), 'Sesgo con (M MXN)']:+.2f} M MXN" for p in periodos]
    lineas.append(f"- **Covariable temporada en {base}** (con vs. sin): " + "; ".join(partes) + ".")
con_respaldo = tabla_regresion.loc[tabla_regresion["Respaldos (%)"] > 0, "Respaldos (%)"]
if len(con_respaldo):
    lineas.append("- **Respaldos naive:** " + ", ".join(f"{m} {v:.0f} %" for m, v in con_respaldo.items())
                  + " de los pronósticos; con esa proporción el modelo no se está evaluando a sí mismo.")
if n_eval < 50:
    lineas.append(f"- ⚠️ **Con {n_eval} pronósticos por modelo las diferencias no son concluyentes:** un acierto más o "
                  f"menos mueve la accuracy {100 / n_eval:.0f} puntos. Estos números validan el pipeline, no eligen ganador.")
display(Markdown("\n".join(lineas)))

### 9.2 Lectura de los datos y del experimento
Con `DEPOSITOS_CARGADOS_FEBAN_sep2025_sep2026_anon.xlsx` (01-09-2025 → 30-09-2026), backtest de 276 orígenes con horizonte de 5 días y la covariable `temporada`:

**Sobre los datos**
1. **El archivo se procesa como historia FEBAN con una sola carga por extracto.** Son 4,804 movimientos en 283 días hábiles, y el paso 4.3 encuentra cero duplicados. Ninguna parte del código distingue el origen de las filas, así que el mismo pipeline correría sin cambios con una historia FEBAN real exportada de la misma forma. La hoja `Notas_datos` del Excel documenta qué periodo se completó y con qué reglas.
2. **La tesorería sigue la misma mecánica todo el año.**
   - HSBC cierra en cero cada día mediante barridos.
   - BBVA concentra fondos en HSBC cada dos semanas: 23 traspasos de 17 a 27 M MXN.
   - Los excedentes se invierten en fondos (GIROFONDO) y se rescatan en días de cargo fuerte.
3. **La única señal sistemática es la temporada.** Así se ve el flujo operativo total (`OPERATIVO_TOTAL`):

| Temporada | Días hábiles | Flujo operativo medio diario | Desv. estándar diaria | Días con salida neta |
|---|---|---|---|---|
| Ene–mar (mayor ingreso) | 64 | 3.93 M MXN | 2.91 M | 3 (5 %) |
| Abr–ago (normal) | 109 | 2.64 M MXN | 2.60 M | 8 (7 %) |
| Sep–dic 2025 + sep 2026 (mayor gasto) | 110 | 2.04 M MXN | 3.18 M | 25 (23 %) |

En ene–mar el ingreso diario es 1.39 veces el normal. En la temporada de mayor gasto el gasto diario es 1.99 veces el normal.

4. **El ruido diario es mayor que el efecto estacional.** La desviación estándar diaria ronda 3 M MXN, mientras que el cambio de nivel entre temporadas es de 0.6 a 1.3 M. La autocorrelación a un día es de apenas 0.03. Por eso el mejor pronóstico es una buena estimación del nivel vigente, y anticipar un cambio de nivel vale más que reaccionar rápido al ruido.
5. **Con un año de historia, la covariable solo puede anticipar la entrada a septiembre de 2026.** En cada origen se usa únicamente cuando todas las temporadas del horizonte ya aparecieron en la historia. Así queda cada transición:
   - **Entrada a ene–mar 2026:** el ingreso alto nunca se había visto, así que no se usa la covariable. Se activa a mediados de enero, cuando la historia ya tiene 10 días de esa temporada (`MIN_DIAS_TEMPORADA`).
   - **Regreso a lo normal en abril de 2026:** la temporada normal tampoco se había visto, porque la historia empieza en septiembre. Sin esta regla, el modelo tendría que extrapolar: con solo sep–mar observados, los dos indicadores suman siempre 1 y su efecto se confunde con la constante. La covariable vuelve a activarse a mediados de abril.
   - **Entrada a la temporada de mayor gasto en septiembre de 2026:** el gasto alto ya se vio en sep–dic de 2025 y la temporada normal en abr–ago, así que la covariable sí puede anticipar el cambio desde los pronósticos hechos a fines de agosto. Por eso la sección 6.3 separa las métricas en el 01-09-2026.

**Sobre el experimento**

6. **Configuración.** Con `HORIZONTE = 5` y `MAX_ORIGENES = 280` hay 276 orígenes posibles (del 03-09-2025 al 23-09-2026), es decir, 5,520 pronósticos por modelo. Los primeros orígenes tienen muy poca historia (desde 3 días), lo que eleva el error de las primeras semanas para todos los modelos. La tabla de MAE mensual permite ver ese arranque por separado.
7. **El periodo desde el 01-09-2026 es pequeño.** Tiene 400 pronósticos por modelo (22 días × 4 series, horizontes 1 a 5), hechos desde 22 orígenes entre el 25-08 y el 23-09, contra 5,120 del periodo anterior. Los pronósticos a 5 días hechos a fines de agosto ya caen en septiembre: ahí se ve si un modelo anticipa el cambio. Con esa muestra, una diferencia de pocos puntos porcentuales de MAE no es concluyente. Conviene leerla junto con el sesgo (tabla 6.3 y gráfica 8.2.1) y con la transición de abril en la tabla mensual.
8. **Cómo leer el efecto de la covariable.** Desde el corte, si el cambio de MAE es negativo y el sesgo se acerca a cero, el modelo anticipa. Si el sesgo de la versión sin covariable es positivo (sobreestima) y se corrige después de unas semanas, ese modelo detecta el cambio tarde. Cada modelo usa la covariable de forma distinta:
   - **Chronos-2** infiere en el contexto la relación entre la covariable y el flujo, sin reentrenarse.
   - **SARIMAX** la estima como un coeficiente de regresión.
9. **Costo de cómputo.** En el backtest, SARIMAX re-estima sus parámetros cada `SARIMAX_REESTIMAR_CADA = 5` orígenes; la latencia de la sección 7 sí se mide estimando desde cero. Agregar la covariable duplica aproximadamente el costo de SARIMAX, y en CPU la corrida completa tarda del orden de 15 a 30 minutos. En Chronos-2, la covariable añade dos variables por serie al lote: eleva su latencia, pero la deja muy por debajo de la de SARIMAX.
10. **No todas las métricas de clasificación son igual de confiables.** El evento *déficit* tiene 508 casos reales de 5,120 antes del corte (10 %) y 53 de 400 desde el corte (13 %). El evento *dirección* está balanceado (50 % y 52 %) y es el más confiable para comparar periodos.
11. **`NETO_TOTAL` está dominado por los traspasos BBVA → HSBC.** Son decisiones de tesorería, no patrones, y ninguna de las dos covariables los describe.
12. **Alcance del resultado.** En esta historia, las temporadas siguen reglas fijas (ver `Notas_datos`) que coinciden exactamente con los meses de la covariable. Por eso la comparación muestra si cada modelo aprovecha una señal de calendario cuando esa señal existe y está bien definida. Con historia real, la misma comparación serviría para comprobar si esas temporadas de verdad existen y con qué intensidad.

**Siguientes pasos**
- Cuando haya historia real, repetir este pipeline sin cambios. Si la covariable deja de ayudar, las temporadas no son tan marcadas como se supuso.
- Agregar en `future_df` covariables de tesorería programada (traspasos BBVA → HSBC y vencimientos de cargos por cartera) para `NETO_TOTAL` y los días de cargo.
- Con dos o más años de historia, comparar la covariable contra dejar que los modelos aprendan solos la estacionalidad anual, y evaluar el ajuste fino de Chronos-2 (`Chronos2Pipeline.fit`).